### Import

In [ ]:
import os
import pandas as pd
import numpy as np
import time
import requests
import json
from huggingface_hub import login
from tqdm import tqdm
import re
import tiktoken
from pathlib import Path

In [ ]:
import torch 
import random

In [ ]:
import os
from openai import OpenAI
client = OpenAI()

### Prompt

In [ ]:
instruction = """  
### TASK
You are an expert reasoning-state annotator.  
Your task is to analyze a chain-of-thought reasoning trace, broken into discrete text sentences, and label each sentence with one or more *LABEL CODES* that describe what this sentence is *doing* functionally in the reasoning process.

### AVAILABLE LABELS and LABEL CODES
Each sentence may express one of the following tasks:

1. initialization (init) → Restating, reframing, or clarifying the task before starting reasoning.
2. deduction (deduce) → Performing calculations, making inferences, drawing implications, doing computations, often doing intermediate conclusions. 
3. augmentation (augment) → Strengthening or extending reasoning through any of the following:
   - recalling facts, internal knowledge, provided information (augment-fact)
   - stating or planning the solution (augment-plan)
   - doing example testing or case trials (augment-test)
   - exploring alternate solution paths (augment-branch)
   - refining/correcting/verifying/merging previous steps, checking calculations, summarizing key points, preparing final answee, re-confirming its own reasoning (augment-refine)
   - using strategies not listed above (augment-emerge)

4. uncertainty estimation (uncertain) → Explicitly expressing doubt, confusion, hesitation, or lack of confidence about current step, assumptions, calculations. May acknowledge missing information, ambiguous problem framing, needs review or the possibility that deduction could be wrong.
5. backtracking (backtrack) → Revisiting earlier steps/assumptions, re-reading the instructions, re-evaluating earlier results often following an uncertainty or realization of error.
6. completion (closure) → Deciding a *final answer/option* to the query or output action, either directly or as a meta-statement.

### ANNOTATION RULES
- Annotate the whole reasoning chain. Label each sentence separately.
- Match which definition best fits the sentence. Assign one label per sentence. Do *NOT* merge sentences.
- The first sentence may *NOT* always be a "init" state. It depends on the label definition.
- The last sentence is usually a "closure" state.
- "init" and "closure" should appear *at most once*.
- If a sentence in the *middle of a chain* expresses a final answer or meta-statement, label it as "deduce" instead. 
- Output only the labeled sentences, preserving exact original text: no text is omitted or paraphrased.
- Do not confuse *backtracking* with *uncertainty-estimation* or *augmentation*.
- Use the exact label codes: init, deduce, augment-subcategory, uncertain, backtrack, closure.
- Always format augmentation as: ["augment-fact"], ["augment-plan"], etc.

### OUTPUT FORMAT
For each sentence, wrap the text as:

["label"]
<sentence>
["end"]

### EXAMPLE
**Input:**
Multiply equation 2 with equation 3.Thus our earlier derivation leading to quartic equation is correct. Simplifying the expression using substituition.Thus polynomial is correct.However, I'm not fully sure. Let me read the question again. I will output the final answer now.

**Expected Output:**
["deduce"] Multiply equation 2 with equation 3. ["end"]
["deduce"] Thus our earlier derivation leading to quartic equation is correct. ["end"]
["deduce"] Simplifying the expression using substituition.["end"]
["deduce"] Thus polynomial is correct. ["end"]   
["uncertain"] However, I'm not fully sure. ["end"]  
["backtrack"] Let me read the question again. ["end"]  
["closure"] I will output the final answer now. ["end"]

Now label each sentence of the given reasoning chain using the annotation rules. No extra text, comments, or metadata.
"""

### Generation

In [ ]:
import tiktoken

# Config
MODEL = ""

# Initialize tokenizer
enc = tiktoken.encoding_for_model(MODEL)

In [ ]:
def count_tokens(text):
    return len(enc.encode(text))

def generate_response(instruction, reasoning_text, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # ensures deterministic cuDNN behavior
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user", "content": reasoning_text}
        ],
        temperature=1e-19,
        seed=seed
    )
    return response

def fix_annotation(reasoning, annotation):
    """
    Fix 'annotation' so that the segment after ["init"] starts
    with the same beginning as 'reasoning'.
    """
    # Extract first word of reasoning
    first_word = reasoning.split()[0]

    # Regex pattern to capture ["init"] ... ["end"]
    pattern = r'(\["init"\]\s*)(.*?)(\s*\["end"\])'
    match = re.search(pattern, annotation, flags=re.DOTALL)

    if match:
        init_text = match.group(2).strip()

        # if the init segment starts with the unwanted "You are trained on data up to..."
        # or doesn't start with the same first word as reasoning, replace it
        if not init_text.startswith(first_word):
            # Replace only the text inside the first ["init"] ... ["end"]
            replacement = f'{match.group(1)}{reasoning.strip()} {match.group(3)}'
            annotation = re.sub(pattern, replacement, annotation, count=1, flags=re.DOTALL)
    return annotation

### Run

In [ ]:
import json
import os

input_file_path = ""
output_file_path = ""

results = []

if os.path.exists(input_file_path):
    with open(input_file_path, "r") as f:
        data_to_annotate = json.load(f)
    print(f"Loaded {len(data_to_annotate)} entries.")
else:
    print("No existing file")

for i in range(len(data_to_annotate)):
    reasoning_text = data_to_annotate[i]['reasoning']

    print(f"\n=== Processing Reasoning Chain {i+1}/{len(data_to_annotate)} ===")
    response = generate_response(instruction, reasoning_text)
    annotation = response.choices[0].message.content
    final_annotation = fix_annotation(reasoning_text, str(annotation))
    data_to_annotate[i]['annotation'] = final_annotation
    results.append(data_to_annotate[i])

    # --------------- SAVE AFTER EACH ITERATION (atomic write) ----------------
    tmp_path = output_file_path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, output_file_path)
    # ------------------------------------------------------------------------

print("Finished all.")